In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

In [14]:
class RewardModel(nn.Module):
    def __init__(self, model_path, use_margin_loss=True):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_path)
        hidden_size = self.backbone.config.hidden_size
        
        self.reward_head = nn.Linear(hidden_size, 1)
        self.use_margin_loss = use_margin_loss
        pass
    
    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids, attention_mask=attention_mask)
        # [batch_size, seq_len, hidden_size]
        last_hidden_state = outputs.last_hidden_state
        
        eos_embdding = last_hidden_state[:,-1,:] # [batch_size, hidden_size]
        reward = self.reward_head(eos_embdding).squeeze(-1) # [batch_size]
        return reward

In [15]:
def pairwise_loss(chosen_rewards, rejected_rewards, margins):
    diff = chosen_rewards-rejected_rewards
    if margins is not None:
        diff = diff-margins
    return -torch.log(torch.sigmoid(diff).mean())

In [16]:
from torch.utils.data import Dataset, DataLoader
class preferenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        prompt, chosen, rejected = self.data[idx]
        
        chosen_enc = self.tokenizer(
            prompt, chosen,
            max_length = self.max_length,
            padding = "max_length",
            truncation = True,
            return_tensors="pt"
        )
        
        rejected_enc = self.tokenizer(
            prompt, rejected,
            max_length = self.max_length,
            padding = "max_length",
            truncation = True,
            return_tensors="pt"
        )
        
        return chosen_enc, rejected_enc
        

In [19]:
model_path = r"C:\Users\Lenovo\Desktop\course_code\chapter2\Qwen\Qwen3-0___6B"
batch_size = 8
grad_accum_steps = 2
use_amp = True
use_margin_loss = True


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RewardModel(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

train_data = [
    ("解释强化学习", "强化学习是智能体通过奖励信号学习决策的方法。", "强化学习是一种算法。"),
    ("牛顿第三定律是什么？", "作用力与反作用力大小相等、方向相反。", "牛顿定律涉及万有引力。"),
]

train_dataset = preferenceDataset(train_data, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


In [22]:
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

model.train()

for epoch in range(10):
    total_loss = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(train_dataloader):
        chosen_batch, rejected_batch = batch
        chosen_inputs = {k:v.squeeze(1) for k,v in chosen_batch.items()}
        rejected_inputs = {k:v.squeeze(1) for k,v in rejected_batch.items()}
        
        with torch.cuda.amp.autocast(enabled=use_amp):
            r_chosen = model(**chosen_inputs)
            r_reject = model(**rejected_inputs)
            margins = torch.rand(len(r_chosen)).to(device) 
            loss = pairwise_loss(r_chosen, r_reject, margins)
        scaler.scale(loss).backward()
        total_loss+=loss.item()
        
        if (step+1)%grad_accum_steps==0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        print(f"Epoch{epoch+1} ,Loss: {total_loss/len(train_dataloader):.4f}")
    
    torch.save(model.state_dict(), "reward_model.pt")
        
                
            

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9612\4186398347.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9612\4186398347.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch1 ,Loss: 1.4305
Epoch2 ,Loss: 1.9370
Epoch3 ,Loss: 1.4068
Epoch4 ,Loss: 1.8195


KeyboardInterrupt: 